# 🥾 Side Trails · Step 1 — More AI Maths
**A Billion Small Steps · Hardware for AI · Step 1 · Book sections App. A (from 1.5, 1.6, 1.7, 1.C) · Foothold lessons 1.5, 1.C**

Read the book sections first, then run this notebook from top to bottom. Parts A–B run anywhere; Parts C–D compare the CPU with a **GPU**: Runtime → Change runtime type → **T4 GPU**.

| Part | What you do | Book |
|---|---|---|
| A | Shapes a network can draw: train a network to find a circle | A.1 (from 1.5) |
| B | Temperature: why the same question gets different answers | A.3 (from 1.C) |
| C | A CNN reads MNIST, on the CPU and on a GPU | A.4 (from 1.6) |
| D | Time attention as the context grows | A.5 (from 1.7) |

⏱️ About 1–2 hours (any part on its own). 📘 Textbook: D2L Ch 4, Ch 7, Ch 11. Course site: https://aishugithub.github.io/course_GPU/

🥾 **Side Trails are optional.** Nothing in a later Core Lab needs them. Each part says where it leaves the Main Climb and where it rejoins.

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import numpy as np                 # NumPy: arrays of numbers and fast maths on them ("np" is its short name)
import matplotlib.pyplot as plt    # Matplotlib: draws charts and pictures ("plt" is its short name)
import time                        # time: we use its stopwatch, time.perf_counter()

# Print arrays neatly: wide lines, 3 decimal places, no 1e-05 style numbers
np.set_printoptions(linewidth=150, precision=3, suppress=True)
print("Ready! NumPy version:", np.__version__)   # show that everything loaded, and NumPy's version

In [ ]:
import torch                                     # PyTorch: networks, on the CPU or the GPU
from torch import nn                             # nn: PyTorch's layers and losses
device = "cuda" if torch.cuda.is_available() else "cpu"   # use the GPU if this session has one
print("device:", device)                         # say which

In [ ]:
def sync():                                          # wait until all queued GPU work has really finished
    if torch.cuda.is_available():                    # (only when there is a GPU)
        torch.cuda.synchronize()                     # wait for it

---
## Part A · Shapes a network can draw  *(leaves from 1.5 → rejoin at 1.5)*

### A2b · How many neurons? Train a network to find a circle
Two neurons gave the B band two edges. A shape with more edges needs more neurons. Here the job is: *which dots lie inside a circle?* We train five networks, with 1, 2, 3, 6 and 32 hidden neurons, and shade where each one says YES. (Real networks use the smooth ReLU bend between layers; we use it here.)

In [ ]:
rng = np.random.default_rng(0)                 # a fixed random-number generator, so everyone gets the same dots
dots = rng.uniform(-1.1, 1.1, size=(600, 2)).astype(np.float32)        # 600 random dots on a square map: (x, y) each
inside = ((dots ** 2).sum(1) < 0.55 ** 2 * 1.9).astype(np.float32)     # label: 1 if a dot is inside the circle, else 0
dots_t = torch.from_numpy(dots)                # the dots as a PyTorch tensor
inside_t = torch.from_numpy(inside)[:, None]   # the labels as a column, the shape the loss expects

g = np.linspace(-1.2, 1.2, 241)                # 241 evenly spaced points along each side of the map
GX, GY = np.meshgrid(g, g)                     # every (x, y) point of a 241 × 241 grid, used to shade the answer
# all the grid points as one long list of (x, y) pairs, so the network can answer for every one at once
grid = torch.tensor(np.c_[GX.ravel(), GY.ravel()], dtype=torch.float32)

def train_circle(h):                           # train a network with h hidden neurons; return (accuracy, network)
    best = None                                # the best network found so far (none yet)
    for trial in range(5):                     # 5 different random starts; keep the best (people do this in practice)
        torch.manual_seed(trial)               # a fixed random start for this trial
        # 2 inputs (x, y) → h hidden neurons with ReLU → 1 output neuron
        net = torch.nn.Sequential(torch.nn.Linear(2, h), torch.nn.ReLU(), torch.nn.Linear(h, 1))
        opt = torch.optim.Adam(net.parameters(), lr=0.03)   # Adam: a smarter version of gradient descent
        for _ in range(1500):                  # 1,500 training steps
            # predict all 600 dots and score the answers (the usual YES/NO loss)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(net(dots_t), inside_t)
            opt.zero_grad()                    # wipe the slopes from the last step
            loss.backward()                    # find the slope of every weight and bias
            opt.step()                         # nudge every weight and bias
        with torch.no_grad():                  # just checking now, not training
            # the fraction of dots it gets right (an output above 0 means YES)
            acc = float(((net(dots_t) > 0).float() == inside_t).float().mean())
        if best is None or acc > best[0]:      # better than the best start so far?
            best = (acc, net)                  # then keep it
    return best                                # the best accuracy and its network

plt.figure(figsize=(12, 2.8))                  # one wide drawing area for five maps
for k, h in enumerate([1, 2, 3, 6, 32]):       # five networks, from 1 hidden neuron to 32
    acc, net = train_circle(h)                 # train it (best of 5 random starts)
    with torch.no_grad():                      # no training here, just answering
        Z = (net(grid) > 0).numpy().reshape(GX.shape)   # the network's YES (1) or NO (0) at every grid point
    plt.subplot(1, 5, k + 1)                   # draw in box number k+1 of 5
    plt.contourf(GX, GY, Z, levels=[0.5, 1.5], colors=["green"], alpha=0.35)    # shade where it says YES
    plt.scatter(dots[:, 0], dots[:, 1], c=np.where(inside > 0, "green", "red"), s=3)   # the dots: green inside, red outside
    plt.title(f"{h} neuron{'s' if h > 1 else ''}: {acc:.0%} right")            # how many neurons, how many dots right
    plt.axis("off")                            # hide the axis numbers
    print(f"{h:2d} hidden neurons: {acc:.1%} of the dots right")              # the same result in words
plt.tight_layout()                             # space the five maps so they don't overlap
plt.show()                                     # display them

With 1 neuron the network gives up and says NO everywhere (61% right, because most dots are outside). 2 neurons make a strip, 3 a rough triangle, and from about 6 neurons the circle is found. With a smooth bend like ReLU a few neurons can make a few extra corners, but the idea is the same: **more neurons, more corners**, and every extra neuron means more multiply-adds.

### ✏️ YOUR TURN (A)
How many hidden neurons did it take before the network found the circle? Why can't 1 or 2 neurons draw it?

*Your answer:*

---
## Part B · Temperature  *(leaves from 1.C → rejoin at 1.C)*
A chatbot does not always pick its most likely next word: it **draws** one at random, using the probabilities as the odds. The **temperature** T divides every score by T before softmax. Four candidate words after *"The exam was …"*: easy, hard, long, cancelled.

In [ ]:
scores = np.array([2.0, 1.0, 0.1, -1.0])         # the four words' scores: easy, hard, long, cancelled

# try three temperatures: low, normal and high
for T in [0.5, 1.0, 2.0]:
    # divide the scores by T, then softmax as before
    p = np.exp(scores / T) / np.exp(scores / T).sum()
    # the four probabilities at this temperature
    print("T =", T, p.round(3))

### ✏️ YOUR TURN (B)
At T = 0.5 and at T = 2.0, how likely is *"cancelled"*? Why can the same question get different answers from a chatbot?

*Your answer:*

---
## Part C · A CNN reads MNIST, on the CPU and on a GPU  *(leaves from 1.6 → rejoin at 1.7)*
Two convolution layers, a max-pool and one ordinary layer to the 10 digits, trained for **one epoch** on all 60,000 images: first on the CPU (a few minutes), then on the GPU.

In [ ]:
from torchvision import datasets, transforms     # MNIST, and a converter from pictures to tensors
from torch.utils.data import DataLoader          # serves the data in batches
to_tensor = transforms.ToTensor()                # pixels 0..255 → floats 0..1
train_data = datasets.MNIST("data", train=True, download=True, transform=to_tensor)    # 60,000 to learn from
test_data = datasets.MNIST("data", train=False, download=True, transform=to_tensor)    # 10,000 to test on
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)   # batches of 64, shuffled every epoch
loss_fn = nn.CrossEntropyLoss()                  # the loss for "which digit?"

In [ ]:
# the CNN: two conv layers, a max-pool, then one ordinary layer to the 10 digits
def make_cnn():
    # the layers, applied one after another
    return nn.Sequential(
        # Conv 1: 1 channel in (grey), 32 filters of 3 × 3 → 32 × 26 × 26
        nn.Conv2d(1, 32, kernel_size=3), nn.ReLU(),
        # Conv 2: 32 channels in, 64 filters → 64 × 24 × 24
        nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(),
        # keep the biggest of every 2 × 2 block → 64 × 12 × 12
        nn.MaxPool2d(2),
        # 64 × 12 × 12 = 9,216 numbers in one row
        nn.Flatten(),
        # an ordinary layer: 9,216 → 10 digit scores
        nn.Linear(64 * 12 * 12, 10),
    # end of the list of layers
    )

# train for one epoch on a device ("cpu" or "cuda") and time it
def run(device):
    # the same starting weights on both devices
    torch.manual_seed(0)
    # build the model and move it to the device
    model = make_cnn().to(device)
    # plain gradient descent, as in Core Lab 3
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    # start the stopwatch
    start = time.perf_counter()
    # one pass over the 60,000 training images, 64 at a time
    for images, labels in train_loader:
        # move this batch to the same device as the model
        images, labels = images.to(device), labels.to(device)
        # 1. predict and 2. measure the loss
        loss = loss_fn(model(images), labels)
        # wipe the old slopes
        optimizer.zero_grad()
        # 3. find the slopes
        loss.backward()
        # 4. nudge every weight
        optimizer.step()
    # a GPU works ahead of Python: wait until it has really finished
    if device == "cuda": torch.cuda.synchronize()
    # seconds for the epoch (then the lab counts the test images it gets right)
    return time.perf_counter() - start

# the same epoch on the CPU, then on the GPU
# the CPU first: one epoch
cpu_s = run("cpu")
# show it
print(f"CPU: 1 epoch in {cpu_s:.1f} s")
# then the GPU, if this session has one
if device == "cuda":
    # a warm-up epoch first: the first CUDA call is slow
    run("cuda")
    # the timed epoch
    gpu_s = run("cuda")
    # the time and the speed-up
    print(f"GPU: 1 epoch in {gpu_s:.1f} s  ({cpu_s / gpu_s:.1f}x faster)")
# no GPU in this session
else:
    # say so
    print("No GPU in this session: switch to a T4 runtime to see the GPU time.")

### ✏️ YOUR TURN (C)
Change Conv 2 to **128** filters (`nn.Conv2d(32, 128, …)` and `nn.Linear(128 * 12 * 12, 10)`). Predict the change in multiply-adds from the formula in Section 1.6 first, then time it.

*Your answer:*

---
## Part D · Time attention as the context grows  *(leaves from 1.7 → rejoin at 1.C)*
Attention in four lines of PyTorch, timed for context lengths 512 to 4,096 (d = 64, one head), on the CPU and the GPU. Each time is the best of 5 calls after a warm-up.

In [ ]:
# attention: scores, softmax, mix the values
def attention(Q, K, V):
    # 1. every query against every key: an (n × n) table
    scores = Q @ K.T / Q.shape[1] ** 0.5
    # 2. softmax along each row: weights that add up to 1
    weights = torch.softmax(scores, dim=1)
    # 3. the weighted mix of the values: (n × n) @ (n × d)
    return weights @ V

# the length of each query, key and value list
d = 64
# the column titles
print("context   score-table numbers    CPU ms    GPU ms")
# double the context each time
for n in [512, 1024, 2048, 4096]:
    # the context and the size of the n × n score table
    row = f"{n:7,}   {n * n:20,}"
    # time it on the CPU, then on the GPU (if there is one)
    for dev in ["cpu", "cuda"] if device == "cuda" else ["cpu"]:
        # random Q, K and V of shape (n, d) on this device
        Q, K, V = (torch.randn(n, d, device=dev) for _ in range(3))
        # the best of 5 timed calls, in milliseconds
        best = float("inf")
        # a warm-up call, then 5 timed ones
        for rep in range(6):
            # wait for any earlier GPU work, then start the stopwatch
            sync() if dev == "cuda" else None; t0 = time.perf_counter()
            # one attention call
            out = attention(Q, K, V)
            # wait until the GPU has really finished (no-op on the CPU)
            sync() if dev == "cuda" else None
            # keep the fastest of the timed calls (rep 0 is the warm-up)
            best = min(best, (time.perf_counter() - t0) * 1000) if rep else best
        # add this device's time to the row
        row += f"   {best:7.2f}"
    # print the row
    print(row)

### ✏️ YOUR TURN (D)
1. Each doubling of the context makes the score table 4× bigger. Does the time grow about 4× per doubling?
2. Predict the GPU time at n = 8,192 from the 4,096 row, then add 8192 to the list and time it. How big is the score table in MB at 4 bytes per number?

*Your answers:*